In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03. PEMC Demo\n",
    "Prediction-Enhanced Monte Carlo for option pricing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.abspath('..'))\n",
    "\n",
    "import numpy as np\n",
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.pipeline.pemc.estimator import PEMCConfig, NeuralPredictor, PEMCEstimator\n",
    "from src.pipeline.pemc.trainer import PEMCTrainer\n",
    "from src.pipeline.pemc.option_models import AsianOptionPricer\n",
    "\n",
    "%matplotlib inline\n",
    "\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"[INFO] Using device: {device}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Create option pricer\n",
    "pricer = AsianOptionPricer()\n",
    "\n",
    "def expensive_sampler(n):\n",
    "    \"\"\"Expensive Monte Carlo sampler\"\"\"\n",
    "    params = {\n",
    "        'S0': 100,\n",
    "        'K': 105,\n",
    "        'T': 1.0,\n",
    "        'r': 0.05,\n",
    "        'sigma': 0.2,\n",
    "        'n_steps': 252\n",
    "    }\n",
    "    payoffs = pricer.payoff(n_sims=n, **params)\n",
    "\n",
    "    # Ensure correct format\n",
    "    if isinstance(payoffs, tuple):\n",
    "        payoffs = payoffs[0]\n",
    "\n",
    "    return np.array(payoffs).reshape(-1, 1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "def cheap_sampler(n):\n",
    "    \"\"\"Cheap feature-only sampler (must correlate with payoff!)\"\"\"\n",
    "    np.random.seed(42)\n",
    "\n",
    "    features = np.column_stack([\n",
    "        np.random.normal(0, 0.1, n),   # Moneyness proxy\n",
    "        np.random.uniform(0.1, 0.4, n),\n",
    "        np.random.uniform(0.01, 0.05, n),\n",
    "        np.random.normal(0, 0.05, n),\n",
    "        np.random.uniform(0.05, 0.2, n),\n",
    "        np.random.uniform(0.8, 1.2, n),\n",
    "        np.random.uniform(0.7, 1.0, n)\n",
    "    ])\n",
    "\n",
    "    return features.astype(np.float32)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialize trainer\n",
    "predictor = NeuralPredictor(input_dim=7).to(device)\n",
    "trainer = PEMCTrainer(predictor, PEMCConfig(), device=device)\n",
    "\n",
    "# Generate training data\n",
    "features, payoffs = trainer.generate_training_data(\n",
    "    sampler=expensive_sampler,\n",
    "    n_samples=10000\n",
    ")\n",
    "\n",
    "features = np.array(features)\n",
    "payoffs = np.array(payoffs).reshape(-1, 1)\n",
    "\n",
    "print(f\"[INFO] Features shape: {features.shape}\")\n",
    "print(f\"[INFO] Payoffs shape: {payoffs.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Train neural predictor\n",
    "history = trainer.train(\n",
    "    features=features,\n",
    "    payoffs=payoffs,\n",
    "    epochs=50,\n",
    "    batch_size=64\n",
    ")\n",
    "\n",
    "if history is None:\n",
    "    print(\"[ERROR] Training failed\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Plot training loss\n",
    "if history:\n",
    "    plt.figure(figsize=(10, 6))\n",
    "    plt.plot(history.get('train_loss', []), label='Train')\n",
    "    plt.plot(history.get('val_loss', []), label='Validation')\n",
    "    plt.xlabel('Epoch')\n",
    "    plt.ylabel('Loss')\n",
    "    plt.title('Neural Predictor Training')\n",
    "    plt.legend()\n",
    "    plt.yscale('log')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"[WARNING] No training history\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Run PEMC estimation\n",
    "estimator = PEMCEstimator(trainer.predictor, PEMCConfig(\n",
    "    n_coupled=1000,\n",
    "    n_independent=100000\n",
    "), device=device)\n",
    "\n",
    "estimate, variance, metrics = estimator.estimate(\n",
    "    expensive_sampler=expensive_sampler,\n",
    "    cheap_sampler=cheap_sampler\n",
    ")\n",
    "\n",
    "print(f\"PEMC Estimate: ${estimate:.4f}\")\n",
    "print(f\"Variance: {variance:.6f}\")\n",
    "print(f\"Variance Reduction: {metrics.get('variance_reduction', 0)*100:.2f}%\")\n",
    "print(f\"95% CI: [${metrics.get('ci_lower', 0):.4f}, ${metrics.get('ci_upper', 0):.4f}]\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Compare with standard MC\n",
    "standard_payoffs = expensive_sampler(10000)\n",
    "\n",
    "standard_estimate = np.mean(standard_payoffs)\n",
    "standard_var = np.var(standard_payoffs) / len(standard_payoffs)\n",
    "\n",
    "print(f\"Standard MC Estimate: ${standard_estimate:.4f}\")\n",
    "print(f\"Standard MC Variance: {standard_var:.6f}\")\n",
    "\n",
    "if 'variance_reduction' in metrics:\n",
    "    print(f\"\\nImprovement: {metrics['variance_reduction']*100:.2f}% variance reduction\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Save predictor\n",
    "os.makedirs('../data/models', exist_ok=True)\n",
    "trainer.save_predictor('../data/models/pemc_predictor.pth')\n",
    "print(\"[INFO] Predictor saved successfully\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
